In [0]:
import os
print(os.getcwd())



In [0]:
%run "../utils/config_loader"

In [0]:
# Cell 1 — Widget
dbutils.widgets.dropdown("environment", "dev", ["dev", "preprod", "prod"], "Environment")



# Cell 3 — Load config
config = load_config_from_widget()
print(f"🚀 Silver Pharmacy Pipeline - {config['environment'].upper()} Environment")
print(f"Status: Running silver transformation...")
print(f"📂 Reading from : {config['catalog']}.{config['schemas']['bronze']}.pharmacy_raw")
print(f"📂 Writing to   : {config['catalog']}.{config['schemas']['silver']}.pharmacy_clean")

# Cell 4 — Environment check
if config['environment'] == 'prod':
    print("⚠️  WARNING: Running on PROD! Make sure preprod is verified!")
elif config['environment'] == 'preprod':
    print("⚠️  WARNING: Running on PREPROD!")
else:
    print("✅ Running on DEV — safe to proceed!")

# Cell 5 — Imports
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Cell 6 — Read from bronze
bronze_table = f"{config['catalog']}.{config['schemas']['bronze']}.pharmacy_raw"
df_bronze = spark.table(bronze_table)
print(f"\n📊 Bronze rows: {df_bronze.count()}")
display(df_bronze)

# Cell 7 — Data type conversion using SQL
df_bronze.createOrReplaceTempView("bronze_temp")

df_typed = spark.sql("""
    SELECT *,
        try_cast(quantity as int) as qty_int,
        try_cast(unit_price as double) as price_double,
        to_date(sale_date, 'yyyy-MM-dd') as sale_date_clean
    FROM bronze_temp
""").drop("quantity", "unit_price", "sale_date") \
  .withColumnRenamed("qty_int", "quantity") \
  .withColumnRenamed("price_double", "unit_price") \
  .withColumnRenamed("sale_date_clean", "sale_date")

print("✅ Data type conversion done!")

# Cell 8 — Data validation BEFORE type conversion
# Cell 8 — Data validation on RAW bronze data (strings)
valid_categories = ["OTC", "Prescription"]
valid_regions    = ["North", "South", "East", "West"]

df_invalid = df_bronze \
    .withColumn("rejection_reason",
        when(col("store_id").isNull(),                   "NULL store_id")
        .when(col("unit_price") == "xyz",                "Invalid unit_price type")
        .when(col("quantity") == "abc",                  "Invalid quantity type")
        .when(col("quantity").cast("int") <= 0,          "Quantity <= 0")
        .when(col("unit_price").cast("double") <= 0,     "Unit price <= 0")
        .when(~col("category").isin(valid_categories),   "Invalid category")
        .when(~col("region").isin(valid_regions),        "Invalid region")
        .otherwise(None)
    ) \
    .filter(col("rejection_reason").isNotNull())

print("\n=== INVALID RECORDS ===")
display(df_invalid.select("sale_id", "store_id", "quantity",
                           "unit_price", "region", "rejection_reason"))
print(f"❌ Invalid records: {df_invalid.count()}")

# Cell 9 — Filter valid records
df_valid = df_typed \
    .filter(col("store_id").isNotNull()) \
    .filter(col("quantity").isNotNull()) \
    .filter(col("unit_price").isNotNull()) \
    .filter(col("quantity") > 0) \
    .filter(col("unit_price") > 0) \
    .filter(col("category").isin(valid_categories)) \
    .filter(col("region").isin(valid_regions))

print(f"✅ Valid records: {df_valid.count()}")

# Cell 10 — Business logic
df_silver = df_valid \
    .withColumn("total_amount",
        round(col("quantity") * col("unit_price"), 2)) \
    .withColumn("is_high_value",
        when(col("total_amount") >= 300, True).otherwise(False)) \
    .withColumn("sale_priority",
        when(col("total_amount") >= 500, "Critical")
        .when(col("total_amount") >= 100, "High")
        .otherwise("Normal")) \
    .withColumn("is_prescription",
        when(col("category") == "Prescription", True).otherwise(False)) \
    .withColumn("ingestion_timestamp", current_timestamp())

display(df_silver)

# Cell 11 — Save to silver
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['catalog']}.{config['schemas']['silver']}")

external_path = f"{config['storage']['external']['silver']}/pharmacy_clean"
silver_table  = f"{config['catalog']}.{config['schemas']['silver']}.pharmacy_clean"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .option("path", external_path) \
    .saveAsTable(silver_table)

print(f"\n=== SILVER COMPLETE ===")
print(f"🌍 Environment  : {config['environment'].upper()}")
print(f"📥 Bronze rows  : {df_bronze.count()}")
print(f"❌ Invalid rows : {df_invalid.count()}")
print(f"✅ Silver rows  : {df_silver.count()}")
print(f"📋 Table        : {silver_table}")
print(f"📂 Path         : {external_path}")